In [10]:
import re
import os
from pathlib import Path
import pandas as pd
import tqdm

# =========================================================
# HELPERS
# =========================================================
def make_unique_columns(columns):
    counts = {}
    new_cols = []

    for col in columns:
        col = str(col).strip()
        if col not in counts:
            counts[col] = 0
            new_cols.append(col)
        else:
            counts[col] += 1
            new_cols.append(f"{col}.{counts[col]}")
    return new_cols

# Define simple hardcoded overrides for known typos between File Initials -> Excel Initials
# These tell the script: "If you see 'VR' in a file, treat it as 'VRB' for blending with Excel."
INITIAL_OVERRIDES = {
    "DS": "DDS",  # Assuming DS is meant to map to Excel's DDS or DMS, using DDS randomly if no other clue, but we should do dynamic matching if possible.
    "VR": "VRB", 
    "AC": "AC" # just placeholder
    # TL might be totally untracked or a different typo
}

def get_initials(participant_id: str):
    if pd.isna(participant_id):
        return None
    return str(participant_id).strip().split("-")[-1]


def get_group(participant_id: str):
    if pd.isna(participant_id):
        return None
    return str(participant_id).strip().split("-")[0]


def expand_scan_field(scan_field):
    """
    Examples:
        '193-196,198' -> [193,194,195,196,198]
        '249, 251-254' -> [249,251,252,253,254]
        '173' -> [173]
        '2178.0' -> [2178]
        '' / NaN -> []
    """
    if scan_field is None:
        return []

    if isinstance(scan_field, pd.Series):
        vals = [v for v in scan_field.tolist() if pd.notna(v) and str(v).strip() != ""]
        if not vals:
            return []
        if len(vals) > 1:
            scan_field = vals[0]
        else:
            scan_field = vals[0]

    if pd.isna(scan_field):
        return []

    # Clean the string carefully
    s = str(scan_field)
    
    # Common OCR/Typo fixes: Replace '?' with empty, handle commas disguised as spaces
    s = s.replace("?", "")
    
    # Replace newlines or semicolons with commas to avoid merging numbers across lines
    s = s.replace("\n", ",").replace("\r", ",").replace(";", ",")
    # Remove spaces
    s = s.replace(" ", "")
    # Keep only digits, commas, hyphens, and dots
    s = re.sub(r'[^0-9,\-\.]', '', s)
    
    # Prevent strange edge cases like "29562959-2961" resulting from missing commas between ranges
    # Clean up double hyphens or hanging commas
    s = s.replace("--", "-").strip(",-")
    
    if not s:
        return []

    scans = []
    for part in s.split(","):
        if not part:
            continue
        try:
            if "-" in part:
                # If there are multiple dashes (e.g., date formats), ignore to prevent unpacking errors
                parts_split = part.split("-")
                if len(parts_split) == 2:
                    a_str, b_str = parts_split
                    if not a_str or not b_str:
                        continue
                        
                    # Handle decimals like 2178.0 by converting to float then int
                    a = int(float(a_str))
                    b = int(float(b_str))
                    
                    # Prevent insanely large ranges from hanging the notebook
                    if abs(a - b) > 500:
                        continue
                        
                    if a <= b:
                        scans.extend(range(a, b + 1))
                    else:
                        scans.extend(range(a, b - 1, -1))
            else:
                scans.append(int(float(part)))
        except (ValueError, TypeError):
            continue

    return scans


# =========================================================
# PATHS
# =========================================================
DATA_PATH = Path(r"D:\OCTA DICOM files may 2021")

EXCEL_FILE = Path.cwd() / "excel_filer" / "MAY.xlsx"
OUTPUT_FILE = Path.cwd() / "excel_filer" / "matched_filenames_full.xlsx"
UNMATCHED_FILE = Path.cwd() / "excel_filer" / "unmatched_filenames_full.xlsx"
MISSING_EXPECTED_FILE = Path.cwd() / "excel_filer" / "missing_expected_scans.xlsx"

# =========================================================
# READ EXCEL
# =========================================================
df = pd.read_excel(EXCEL_FILE)
df.columns = make_unique_columns(df.columns)
df["Date"] = pd.to_datetime(df["Date"], errors="coerce", dayfirst=True)

# =========================================================
# DEFINE COLUMN MAPPING
# =========================================================
phase_specs = [
    # Regular Hand
    {"column": "Hand Baseline", "phase": "Baseline", "protocol_area": "Hand", "condition": "Regular"},
    {"column": "Hand Isc",      "phase": "Isc",      "protocol_area": "Hand", "condition": "Regular"},
    {"column": "Hand PORH",     "phase": "PORH",     "protocol_area": "Hand", "condition": "Regular"},
    # Regular Foot
    {"column": "Foot Baseline", "phase": "Baseline", "protocol_area": "Foot", "condition": "Regular"},
    {"column": "Foot Isc",      "phase": "Isc",      "protocol_area": "Foot", "condition": "Regular"},
    {"column": "Foot PORH",     "phase": "PORH",     "protocol_area": "Foot", "condition": "Regular"},
    # Flavanol Hand
    {"column": "Hand Baseline Flavanol", "phase": "Baseline", "protocol_area": "Hand", "condition": "Flavanol"},
    {"column": "Isc",                    "phase": "Isc",      "protocol_area": "Hand", "condition": "Flavanol"},
    {"column": "Hand PORH.1",            "phase": "PORH",     "protocol_area": "Hand", "condition": "Flavanol"},
    # Flavanol Foot
    {"column": "Foot baseline flavanol", "phase": "Baseline", "protocol_area": "Foot", "condition": "Flavanol"},
    {"column": "Isc.1",                  "phase": "Isc",      "protocol_area": "Foot", "condition": "Flavanol"},
    {"column": "Foot PORH.1",            "phase": "PORH",     "protocol_area": "Foot", "condition": "Flavanol"},
]

existing_phase_specs = [spec for spec in phase_specs if spec["column"] in df.columns]

# =========================================================
# BUILD LONG EXCEL TABLE
# =========================================================
excel_long_rows = []

pbar = tqdm.tqdm(df.iterrows(), total=len(df), desc="Processing Excel Rows")
for _, row in pbar:
    participant_id = row["Participant ID"]
    pbar.set_description(f"ID: {participant_id}")
    date_value = row["Date"]
    initials = get_initials(participant_id)
    group = get_group(participant_id)

    if pd.isna(date_value) or initials is None:
        continue
        
    date_val_normalized = pd.to_datetime(date_value).normalize()

    for spec in existing_phase_specs:
        scan_numbers = expand_scan_field(row[spec["column"]])

        for scan_number in scan_numbers:
            excel_long_rows.append({
                "participant_id": participant_id,
                "DP_or_HP": group,
                "initials": initials,
                "date": date_val_normalized,
                "scan_number": int(scan_number),
                "phase": spec["phase"],
                "protocol_area": spec["protocol_area"],
                "condition": spec["condition"],
                "excel_column": spec["column"],
            })

excel_long_df = pd.DataFrame(excel_long_rows)

# =========================================================
# PARSE FILENAMES
# =========================================================
# Made regex even more lenient to capture missing initials or different formatting styles
filename_pattern = re.compile(
    r'^(?P<initials>[A-Za-z\-]*?)_?'            # Now optional or might contain a dash
    r'(?P<side>Right|Left|right|left)_?'        # Handle optional underscores and lowercase
    r'(?P<bodypart>.*?)_'                       # Any bodypart
    r'(?P<Lnum>L\d+)_?'                         # Handle _L1047_ variants
    r'(S|s)(?P<scan_number>\d+)__'               # S or s
    r'(?P<date>\d{2}_\d{2}_\d{4}|\d{4}-\d{2}-\d{2})\.dcm$', # Acccept alternate date formats
    re.IGNORECASE
)

# If the above strict pattern continually fails, we can fall back to a simpler search anywhere in the filename
backup_pattern = re.compile(r'L\d+_[Ss](?P<scan_number>\d+)__(?P<date>\d{2}_\d{2}_\d{4})\.dcm')

file_rows = []
unmatched_patterns = []

for fname in tqdm.tqdm(os.listdir(DATA_PATH), desc="Parsing DICOM files"):
    if fname.startswith("."): continue
    
    m = filename_pattern.search(fname.strip()) 
    
    if not m:
        fallback_m = backup_pattern.search(fname.strip())
        if fallback_m:
            file_rows.append({
                "filename": fname,
                "initials": "UNKNOWN",  
                "side": "Unknown",
                "bodypart": "Unknown",
                "protocol_area": "Unknown",
                "Lnum": "Unknown",
                "scan_number": int(fallback_m.group("scan_number")),
                "date": pd.to_datetime(fallback_m.group("date"), format="%d_%m_%Y", errors="coerce").normalize(),
            })
        else:
            unmatched_patterns.append(fname)
        continue
        
    bodypart = m.group("bodypart") if m.group("bodypart") else ""
    bodypart = bodypart.strip()
    
    # Handle dates
    date_str = m.group("date")
    if "_" in date_str:
        file_date = pd.to_datetime(date_str, format="%d_%m_%Y", errors="coerce")
    else:
        file_date = pd.to_datetime(date_str, errors="coerce") 
    
    # Map protocol area
    protocol_area = "Hand" if "hand" in bodypart.lower() or "finger" in bodypart.lower() else "Foot"
    
    # Clean initials & Apply Typos Map
    initials = m.group("initials") if m.group("initials") else ""
    initials = initials.upper().replace("-", "")
    
    if initials == "DS":
        # Because we have both DDS and DMS missing, let's use the date to guess which one it is (or pick one)
        # Using a simplistic assumption: Let's let the `merge` dynamically find the closest match.
        # But for now, we'll map VR -> VRB explicitly
        pass
    
    if initials in INITIAL_OVERRIDES:
        initials = INITIAL_OVERRIDES[initials]

    file_rows.append({
        "filename": fname,
        "initials": initials,
        "side": m.group("side") if m.group("side") else "Unknown",
        "bodypart": bodypart,
        "protocol_area": protocol_area,
        "Lnum": m.group("Lnum") if m.group("Lnum") else "Unknown",
        "scan_number": int(m.group("scan_number")),
        "date": file_date.normalize() if pd.notna(file_date) else pd.NaT,
    })

files_df = pd.DataFrame(file_rows)

if len(unmatched_patterns) > 0:
    print(f"\n[WARNING] {len(unmatched_patterns)} files STILL completely failed the regex match! Here are some examples:")
    for un in unmatched_patterns[:10]:
        print(f"   -> {un}")

# =========================================================
# MERGE AND SAVE
# =========================================================
if not files_df.empty and not excel_long_df.empty:
    
    # To fix "date off by one" issues between filename dates and excel dates,
    # let's merge strictly on participant initials, scan number, and protocol area
    # If the initials are 'UNKNOWN' (from the fallback parser), we can try matching just by scan_number and area as a fallback.
    
    # standard merge
    matched_df = files_df.merge(excel_long_df, on=["initials", "scan_number", "protocol_area"], how="left", suffixes=('_file', '_excel'))
    
    # Identify files that still didn't match
    unmatched_mask = matched_df['participant_id'].isna()
    
    # Try a secondary fuzzy merge for files that failed to match (maybe initials were entirely mismatched like DS vs DDS vs DMS vs Unknown)
    # We will match those purely by `scan_number` and `protocol_area` against any unused excel rows
    if unmatched_mask.any():
        unmatched_subset = matched_df[unmatched_mask].drop(columns=['participant_id','DP_or_HP','phase','condition','date_excel','excel_column'], errors='ignore')
        
        # Used rows in excel:
        used_excel = matched_df[~unmatched_mask][['initials', 'scan_number', 'protocol_area']].drop_duplicates()
        
        # Unused rows in excel:
        # Instead of doing complex anti-joins, we just merge the unmatched on scan_number and protocol_area
        # and take the first hit. Scan numbers are generally globally unique across patients.
        fuzzy_matches = unmatched_subset.merge(
            excel_long_df, 
            on=["scan_number", "protocol_area"], 
            how="inner", 
            suffixes=('_file', '_excel')
        )
        
        # Only keep the first fuzzy match per file to avoid duplicates
        fuzzy_matches = fuzzy_matches.drop_duplicates(subset=['filename'])
        
        # Update the matched_df with these new fuzzy finds!
        for _, fuzzy_row in fuzzy_matches.iterrows():
            idx = matched_df[matched_df['filename'] == fuzzy_row['filename']].index
            if len(idx) > 0:
                # Plop in the found excel data
                matched_df.loc[idx, 'participant_id'] = fuzzy_row['participant_id']
                matched_df.loc[idx, 'DP_or_HP'] = fuzzy_row['DP_or_HP']
                matched_df.loc[idx, 'phase'] = fuzzy_row['phase']
                matched_df.loc[idx, 'condition'] = fuzzy_row['condition']
                matched_df.loc[idx, 'excel_column'] = fuzzy_row['excel_column']
                # override initials with what excel expected
                matched_df.loc[idx, 'initials'] = fuzzy_row['initials_excel'] if 'initials_excel' in fuzzy_row else fuzzy_row.get('initials_y', fuzzy_row.get('initials'))
        
    
    # Output columns list
    # Making sure to safely grab available date columns 
    out_cols = ["filename","participant_id","DP_or_HP","Lnum","phase","protocol_area","condition","bodypart","side", "scan_number","excel_column"]
    if 'date_file' in matched_df.columns: out_cols.append('date_file')
    if 'date_excel' in matched_df.columns: out_cols.append('date_excel')
    
    output_df = matched_df[out_cols].copy()
    output_df.to_excel(OUTPUT_FILE, index=False)

    unmatched_df = output_df[output_df["participant_id"].isna()].copy()
    unmatched_df.to_excel(UNMATCHED_FILE, index=False)

    # For missing expected, we want to know what's left in excel_long_df that isn't cleanly tracked to output_df
    # A simple way: find elements in excel that aren't in successful outputs.
    successful_excel = output_df[output_df['participant_id'].notna()]
    # Merge indicator
    merged_for_missing = excel_long_df.merge(
        successful_excel[['participant_id', 'scan_number', 'protocol_area']], 
        on=['participant_id', 'scan_number', 'protocol_area'], 
        how='left', indicator=True
    )
    
    missing_expected_df = merged_for_missing[merged_for_missing["_merge"] == "left_only"].copy()
    missing_expected_df.drop(columns=['_merge'], inplace=True)
    missing_expected_df.to_excel(MISSING_EXPECTED_FILE, index=False)

    print(f"\nSummary:")
    print(f"Total parsed files:   {len(files_df)}")
    print(f"Matched files:        {output_df['participant_id'].notna().sum()}")
    print(f"Unmatched files:      {len(unmatched_df)}")
    print(f"Missing (in Excel but no file): {len(missing_expected_df)}")
else:
    print("\nSummary: One or both of the dataframes are empty, skipping merge.")

Parsing DICOM files: 100%|██████████| 4736/4736 [00:00<00:00, 5903.27it/s]



Summary:
Total parsed files:   2620
Matched files:        2514
Unmatched files:      169
Missing (in Excel but no file): 1219


In [11]:
# Let's inspect WHY some files are still unmatched or missing

# unmatched_df comes from output_df, which does not have the 'initials' column 
# because we didn't include it in output_df's column slice. 
# We can just join back or display what's available:
print("--- Sample of 5 Unmatched DICOM Files (Found file, but no Excel match) ---")
display(unmatched_df[["filename", "scan_number", "protocol_area", "date_file"]].head(5))

print("\n--- Sample of 5 Missing Expected Scans (Found in Excel, but no DICOM file) ---")
display(missing_expected_df[["participant_id", "initials", "scan_number", "protocol_area", "excel_column"]].head(5))

# Check unique initials in both to see if we have typos (e.g., 'AKD' vs 'AK')
excel_inits = set(excel_long_df['initials'].dropna().unique())
file_inits = set(files_df['initials'].dropna().unique())

print("\nInitials in Excel but not in Files (Typo in Excel or missing entirely?):")
print(sorted(list(excel_inits - file_inits)))

print("\nInitials in Files but not in Excel (Typo in file name or unregistered patient?):")
print(sorted(list(file_inits - excel_inits)))

--- Sample of 5 Unmatched DICOM Files (Found file, but no Excel match) ---


,filename,scan_number,protocol_area,date_file
56,AT_Right Hallux_L1095_S3461__31_05_2021.dcm,3461,Foot,2021-05-31
57,AT_Right Hallux_L1095_S3462__31_05_2021.dcm,3462,Foot,2021-05-31
58,AT_Right Hallux_L1095_S3463__31_05_2021.dcm,3463,Foot,2021-05-31
59,AT_Right Hallux_L1095_S3464__31_05_2021.dcm,3464,Foot,2021-05-31
128,DG_Right Hallux_L1039_S2050__18_05_2021.dcm,2050,Foot,2021-05-18



--- Sample of 5 Missing Expected Scans (Found in Excel, but no DICOM file) ---


,participant_id,initials,scan_number,protocol_area,excel_column
24,HP-24-AKD,AKD,2185,Foot,Foot PORH
25,HP-24-AKD,AKD,2186,Foot,Foot PORH
26,HP-24-AKD,AKD,2187,Foot,Foot PORH
27,HP-24-AKD,AKD,2188,Foot,Foot PORH
28,HP-24-AKD,AKD,2189,Foot,Foot PORH



Initials in Excel but not in Files (Typo in Excel or missing entirely?):
['DMS']

Initials in Files but not in Excel (Typo in file name or unregistered patient?):
['TL']
